In [35]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pm4py
from IPython.display import display


ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) 
from src.data import *
from src import censored_petri as cp

pm4py.util.constants.SHOW_PROGRESS_BAR = False

### 1. Разметка сессий и анализ исходов

Разделение Журнала событий на сессии и определение исхода каждой сессии.

  - Сессии размечаются по событиям `Kernel-General:12` (старт)
  - Возможные исходы: `FAIL` (если в начале следующей сессии есть `Kernel-Power:41`) или `CENSORED` (штатное завершение или продолжающаяся сессия)
  
- статистика:

  - Общее число сессий и распределение по исходам (FAIL/CENSORED)
  - Сводная таблица по машинам: сколько сессий с каждым исходом
  - таблица: связь исхода с наличием штатной остановки (`has_clean_stop`) и последней сессии (`is_last`)
  - Статистика длительности сессий в часах для FAIL и CENSORED
  - Число сессий нулевой длины


In [36]:
df, sessions = mark_sessions(load_dataset())

print("Kernel-Power:41 всего:", (df["pair"] == K41).sum(),
      "| отнесено к предыдущей сессии:", df["marks_prev_fail"].sum())
print("Сессий всего:", len(sessions))
print(sessions["outcome"].value_counts())
print(sessions.pivot_table(index="machine_id", columns="outcome", values="session_no",
                           aggfunc="count", fill_value=0))
print(pd.crosstab(sessions["outcome"], [sessions["has_clean_stop"], sessions["is_last"]]))

hours = (sessions["t_end"] - sessions["t_start"]).dt.total_seconds() / 3600
print(hours.groupby(sessions["outcome"]).describe().round(2))
print("Сессий нулевой длины:", (hours == 0).sum())

Kernel-Power:41 всего: 300 | отнесено к предыдущей сессии: 294
Сессий всего: 955
outcome
CENSORED    661
FAIL        294
Name: count, dtype: int64
outcome     CENSORED  FAIL
machine_id                
pc_01              8    94
pc_02             46     1
pc_03             66     1
pc_04             58     0
pc_05             58     0
pc_06             73     4
pc_07            209     3
pc_08             24   103
pc_09              9     0
pc_10              1     4
pc_11              1     0
pc_12             12     0
pc_13             40    37
pc_14             23     1
pc_15             17    32
pc_16             16    14
has_clean_stop False       True 
is_last        False True  False
outcome                         
CENSORED           0    16   645
FAIL             294     0     0
          count    mean     std  min   25%    50%    75%      max
outcome                                                          
CENSORED  661.0   43.77  356.19  0.0  0.04   0.48  10.02  5829.56
FAIL

### 2. Журналы компонент (Kernel-Power)
Строится журнал событий для компоненты `Kernel-Power`.

- сводка по машинам:
  - `sessions_all` — всего сессий у машины
  - `sessions_in_log` — сессий с событиями компоненты
  - `events` — число событий компоненты
  - `fail` — число отказов в журнале
  - `share_empty` — доля пустых сессий (убраны из журнала)
  - `events_per_session` — среднее число событий на сессию
- Выводится распределение событий


(данные сохраняются в `data/processed/component_logs/kernel_power.csv`)


Машины pc_10, pc_11 без событий компоненты, далее не учитываются

In [37]:
KP = "Microsoft-Windows-Kernel-Power"
kp_log = build_component_log(df, sessions, KP, drop_pairs=(K41,))
print(summarize_component_log(kp_log, sessions))

ev = kp_log[~kp_log["activity"].str.startswith("END_")]
print(ev["activity"].value_counts(normalize=True).head(10).round(3))
print(save_component_log(kp_log, "kernel_power").relative_to(ROOT))

            sessions_all  sessions_in_log  events  fail  share_empty  \
machine_id                                                             
pc_01                102              102     376    94          0.0   
pc_02                 47               47      92     1          0.0   
pc_03                 67               67     132     1          0.0   
pc_04                 58               58     115     0          0.0   
pc_05                 58               58     115     0          0.0   
pc_06                 77               77     149     4          0.0   
pc_07                212              212     629     3          0.0   
pc_08                127              127     800   103          0.0   
pc_09                  9                9     615     0          0.0   
pc_10                  5                0       0     0          1.0   
pc_11                  1                0       0     0          1.0   
pc_12                 12               12     228     0         

### 3. Таблица переходов и анализ вероятностей

строим таблицу переходов между состояниями компоненты и с анализом вероятности перехода одного состояния в другое.

**Что делается:**
- Вызывается `transition_table()`, которая для каждой сессии строит цепочку `START → события → END_*`
- Выводится:
  - Общее число уникальных переходов
  - Какие переходы встречаются чаще всего по количеству (`count`)
  - Какие переходы более вероятны (`prob`), где `count >= 10`
  - Из каких состояний чаще всего совершаются переходы в конечные состояния (`END_CENSORED`, `END_FAIL`)


In [38]:
from src.data import load_dataset, mark_sessions, build_component_log
from src.models import transition_table

df, sessions = mark_sessions(load_dataset())

KP = "Microsoft-Windows-Kernel-Power"
K41 = "Microsoft-Windows-Kernel-Power:41"
kp_log = build_component_log(df, sessions, KP, drop_pairs=(K41,))

tt_all = transition_table(kp_log)

print(f"Всего уникальных переходов: {len(tt_all)}")

print("\nТоп-10 переходов по количеству (count):")
print(tt_all.nlargest(10, "count").to_string(index=False))

print("\nТоп-10 переходов по вероятности (prob), где count >= 10:")

tt_significant = tt_all[tt_all["count"] >= 10]
print(tt_significant.nlargest(10, "prob").to_string(index=False))

print("\nПереходы в конечные состояния (END_CENSORED / END_FAIL):")
end_transitions = tt_all[tt_all["dst"].str.startswith("END_")]
print(end_transitions.sort_values("count", ascending=False).head(10).to_string(index=False))

Всего уникальных переходов: 91

Топ-10 переходов по количеству (count):
src dst  count     prob
566 506   5011 0.431574
507 566   4959 0.999597
506 566   3467 0.691878
566 566   2237 0.192662
566 507   2069 0.178193
506 507   1366 0.272600
 42 107   1084 1.000000
566 172   1036 0.089226
172 507    972 0.326613
566 105    862 0.074240

Топ-10 переходов по вероятности (prob), где count >= 10:
  src          dst  count     prob
   42          107   1084 1.000000
  577 END_CENSORED     95 1.000000
  507          566   4959 0.999597
  131          130    369 0.960938
  130           42    329 0.891599
  109 END_CENSORED    550 0.852713
START          172    797 0.839831
  578          521     25 0.781250
  506          566   3467 0.691878
  187           42    320 0.645161

Переходы в конечные состояния (END_CENSORED / END_FAIL):
src          dst  count     prob
109 END_CENSORED    550 0.852713
107     END_FAIL    134 0.123616
577 END_CENSORED     95 1.000000
172     END_FAIL     77 0.02587

In [39]:
# Добавляем START в начало каждой сессии (в памяти, без сохранения на диск)
starts = kp_log.groupby(["machine_id", "session_no"], as_index=False).agg(
    case_id=("case_id", "first"),
    machine_id=("machine_id", "first"),
    session_no=("session_no", "first"),
    time=("time", "min")
)
starts["activity"] = "START"
starts["seq"] = -1
starts["time"] = starts["time"] - pd.Timedelta(seconds=1)

kp_log = pd.concat([starts[kp_log.columns], kp_log], ignore_index=True)
kp_log = kp_log.sort_values(["machine_id", "session_no", "seq"]).reset_index(drop=True)
print(f"Добавлено START. Всего строк: {len(kp_log)}")

Добавлено START. Всего строк: 33394


### 4. Сохранение схем по машинам и построение сетей Петри

Сохраняем таблицы переходов для каждой машины в JSON и строим сети Петри.

машины с малым числом сессий/событий исключаются


In [40]:
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"
from src.models import *

COMPONENT = "kernel_power"

print("Сохранение JSON-схем переходов по машинам ===")
schemes_dir = save_machine_schemes(kp_log, COMPONENT)
print(f"Схемы сохранены в: {schemes_dir}")

print("Построение локальных сетей Петри")
# Пороги: 20 сессий и 50 строк журнала, включая конечные активности.
petri_dir, valid_m, excluded_m, local_models = save_machine_petri_nets(
    kp_log, COMPONENT, min_sessions=20, min_events=50
)
print(f"Картинки сетей Петри сохранены в: {petri_dir}")
print("Машины для сравнения:", valid_m)
print("Исключены:", excluded_m)


Сохранение JSON-схем переходов по машинам ===
Схемы сохранены в: C:\Users\lena\Desktop\diplom\EquipmentEventLogGeneration\results\schemes\kernel_power
Построение локальных сетей Петри
Всего машин в логе: 14
Будет построено сетей Петри: 12 (сессий >= 20, событий >= 50)
 Исключены (мало данных): ['pc_09', 'pc_12']
Картинки сетей Петри сохранены в: C:\Users\lena\Desktop\diplom\EquipmentEventLogGeneration\results\petri_nets\kernel_power
Машины для сравнения: ['pc_01', 'pc_02', 'pc_03', 'pc_04', 'pc_05', 'pc_06', 'pc_07', 'pc_08', 'pc_13', 'pc_14', 'pc_15', 'pc_16']
Исключены: ['pc_09', 'pc_12']


In [41]:
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

print("=== Глобальная сеть Петри для Kernel-Power ===")
global_net, global_im, global_fm, out_dir = build_global_petri_net(
    kp_log, COMPONENT, min_sessions=20, min_events=50
)


=== Глобальная сеть Петри для Kernel-Power ===
Глобальная сеть: машин в работе 12 из 14
  Суммарно сессий: 212, событий: 32509
Сохранено: C:\Users\lena\Desktop\diplom\EquipmentEventLogGeneration\results\petri_nets\kernel_power\global.pnml
Сохранено: C:\Users\lena\Desktop\diplom\EquipmentEventLogGeneration\results\petri_nets\kernel_power\global.png


In [42]:
print("Число case_id:", kp_log["case_id"].nunique())
print("Число пар машина-сессия:",
      kp_log[["machine_id", "session_no"]].drop_duplicates().shape[0])

print(
    kp_log.groupby(["machine_id", "session_no"])["case_id"]
    .nunique()
    .value_counts()
)

Число case_id: 949
Число пар машина-сессия: 949
case_id
1    949
Name: count, dtype: int64


### 5. Метрики качества локальных и глобальной сетей Петри

Для оценки используются метрики PM4Py: **token-based fitness** и **token-based precision**. 


`END_FAIL` и `END_CENSORED` - обычные конечные активности для стандартных метрик PM4Py, они не учитывают цензурирование статистически (как в 5ом ноутбуке)

- **5.0** — подготовка журналов и настройки

In [43]:
from pm4py.objects.log.obj import Event, EventLog, Trace
from time import perf_counter

# Переключатели вывода и расчётов
SHOW_PROGRESS = False     #показывать/скрывать сообщения о ходе расчёта
COMPUTE_LOCAL_PRECISION = True  #считать/пропускать локальный precision
COMPUTE_MATRIX_FITNESS = True   #считать/пропускать матрицу fitness
COMPUTE_GLOBAL_PRECISION = True #считать/пропускать precision глобальной модели

def progress(message):
    if SHOW_PROGRESS:
        print(message, flush=True)

def timed_start(label):
    progress(f"\n[{label}] начало")
    return perf_counter()

def timed_end(label, started):
    progress(f"[{label}] завершено за {perf_counter() - started:.1f} сек.")

def to_pm4py_log(frame):
    """Преобразует журнал одной машины в EventLog, сохраняя порядок seq."""
    frame = frame.sort_values(["machine_id", "session_no", "seq"])
    traces = []
    for (machine_id, session_no), group in frame.groupby(["machine_id", "session_no"], sort=False):
        trace = Trace(
            [Event({"concept:name": str(activity)}) for activity in group["activity"]],
            attributes={"concept:name": f"{machine_id}_{session_no}", "machine_id": str(machine_id)},
        )
        if len(trace):
            traces.append(trace)
    return EventLog(traces)

def get_fitness(event_log, model):
    net, im, fm = model
    result = pm4py.fitness_token_based_replay(event_log, net, im, fm)
    return {
        "fitness": result.get("log_fitness"),
        "average_trace_fitness": result.get("average_trace_fitness"),
        "percentage_fitting_traces": result.get("percentage_of_fitting_traces"),
    }

def get_precision(event_log, model):
    net, im, fm = model
    return float(pm4py.precision_token_based_replay(event_log, net, im, fm))

# Журналы преобразуются один раз и затем используются всеми этапами.
valid_m = list(local_models)
started = timed_start("Подготовка журналов")
machine_logs = {
    machine_id: to_pm4py_log(kp_log[kp_log["machine_id"] == machine_id])
    for machine_id in valid_m
}
timed_end("Подготовка журналов", started)
progress(f"Машин: {len(machine_logs)}; трасс: {sum(len(log) for log in machine_logs.values())}")


- **5.1** — fitness локальной модели на журнале своей машины

In [44]:
started = timed_start("Локальный fitness")
local_fitness_rows = []
for i, machine_id in enumerate(valid_m, 1):
    progress(f"Локальный fitness: {i}/{len(valid_m)} — {machine_id}")
    fit = get_fitness(machine_logs[machine_id], local_models[machine_id])
    local_fitness_rows.append({"machine_id": machine_id, **fit})
local_fitness = pd.DataFrame(local_fitness_rows).set_index("machine_id")
display(local_fitness.round(4))
timed_end("Локальный fitness", started)


,fitness,average_trace_fitness,percentage_fitting_traces
machine_id,,,
pc_01,1.0,1.0,100.0
pc_02,1.0,1.0,100.0
pc_03,1.0,1.0,100.0
pc_04,1.0,1.0,100.0
pc_05,1.0,1.0,100.0
pc_06,1.0,1.0,100.0
pc_07,1.0,1.0,100.0
pc_08,1.0,1.0,100.0
pc_13,1.0,1.0,100.0



- **5.2** — precision локальной модели

In [45]:
started = timed_start("Локальный precision")
local_precision_rows = []
if COMPUTE_LOCAL_PRECISION:
    for i, machine_id in enumerate(valid_m, 1):
        progress(f"Локальный precision: {i}/{len(valid_m)} — {machine_id}")
        precision = get_precision(machine_logs[machine_id], local_models[machine_id])
        local_precision_rows.append({"machine_id": machine_id, "precision": precision})
else:
    progress("Пропускаю локальный precision: COMPUTE_LOCAL_PRECISION=False")
local_precision = pd.DataFrame(local_precision_rows).set_index("machine_id") if local_precision_rows else pd.DataFrame(index=valid_m, columns=["precision"])
display(local_precision.round(4))
timed_end("Локальный precision", started)


,precision
machine_id,
pc_01,0.8977
pc_02,1.0000
pc_03,1.0000
pc_04,1.0000
pc_05,1.0000
pc_06,1.0000
pc_07,0.7988
pc_08,0.7386
pc_13,0.1987


- **5.3** — матрица fitness: строки — проверяемые журналы, столбцы — локальные модели.


In [46]:
started = timed_start("Матрица fitness")
fitness_matrix = pd.DataFrame(index=valid_m, columns=valid_m, dtype=float)
if COMPUTE_MATRIX_FITNESS:
    total = len(valid_m) * len(valid_m)
    done = 0
    for log_machine in valid_m:
        for model_machine in valid_m:
            done += 1
            progress(f"Fitness matrix: {done}/{total} — журнал {log_machine} / модель {model_machine}")
            fitness_matrix.loc[log_machine, model_machine] = get_fitness(
                machine_logs[log_machine], local_models[model_machine]
            )["fitness"]
    display(fitness_matrix.round(4))
else:
    progress("Пропускаю матрицу fitness: COMPUTE_MATRIX_FITNESS=False")
timed_end("Матрица fitness", started)


,pc_01,pc_02,pc_03,pc_04,pc_05,pc_06,pc_07,pc_08,pc_13,pc_14,pc_15,pc_16
pc_01,1.0000,1.0000,1.0000,0.7081,0.7081,1.0000,1.0000,1.0000,0.7121,0.6163,0.9261,0.7367
pc_02,1.0000,1.0000,1.0000,0.9957,0.9957,1.0000,1.0000,1.0000,0.6726,0.5036,1.0000,0.5036
pc_03,1.0000,1.0000,1.0000,0.9970,0.9970,1.0000,1.0000,1.0000,0.6708,0.5025,1.0000,0.5025
pc_04,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.6695,0.5014,1.0000,0.5014
pc_05,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.6695,0.5014,1.0000,0.5014
pc_06,1.0000,1.0000,1.0000,0.9894,0.9894,1.0000,1.0000,1.0000,0.6747,0.5055,1.0000,0.5055
pc_07,1.0000,1.0000,1.0000,0.9972,0.9972,1.0000,1.0000,1.0000,0.7158,0.6680,1.0000,0.6680
pc_08,0.9993,1.0000,1.0000,0.7599,0.7599,1.0000,1.0000,1.0000,0.7332,0.7125,0.9449,0.7832
pc_13,0.9779,1.0000,1.0000,0.8810,0.8810,1.0000,0.8188,0.9948,1.0000,0.9789,1.0000,0.9928
pc_14,0.0777,0.0534,0.0534,0.0524,0.0524,0.0534,0.0574,0.1141,0.9442,1.0000,0.6062,1.0000


- **5.4.** Fitness глобальной модели на журнале каждой машины

In [47]:

started = timed_start("Глобальный fitness")
global_model = (global_net, global_im, global_fm)
global_fitness_rows = []
for i, machine_id in enumerate(valid_m, 1):
    progress(f"Глобальный fitness: {i}/{len(valid_m)} — {machine_id}")
    fit = get_fitness(machine_logs[machine_id], global_model)
    global_fitness_rows.append({"machine_id": machine_id, **fit})
global_fitness = pd.DataFrame(global_fitness_rows).set_index("machine_id")
display(global_fitness.round(4))
timed_end("Глобальный fitness", started)


,fitness,average_trace_fitness,percentage_fitting_traces
machine_id,,,
pc_01,1.0,1.0,100.0
pc_02,1.0,1.0,100.0
pc_03,1.0,1.0,100.0
pc_04,1.0,1.0,100.0
pc_05,1.0,1.0,100.0
pc_06,1.0,1.0,100.0
pc_07,1.0,1.0,100.0
pc_08,1.0,1.0,100.0
pc_13,1.0,1.0,100.0


**5.5.** Precision глобальной модели

In [48]:
started = timed_start("Глобальный precision")
global_precision_rows = []
if COMPUTE_GLOBAL_PRECISION:
    for i, machine_id in enumerate(valid_m, 1):
        progress(f"Глобальный precision: {i}/{len(valid_m)} — {machine_id}")
        precision = get_precision(machine_logs[machine_id], global_model)
        global_precision_rows.append({"machine_id": machine_id, "precision": precision})
else:
    progress("Пропускаю глобальный precision: COMPUTE_GLOBAL_PRECISION=False")
global_precision = pd.DataFrame(global_precision_rows).set_index("machine_id") if global_precision_rows else pd.DataFrame(index=valid_m, columns=["precision"])
display(global_precision.round(4))
timed_end("Глобальный precision", started)


,precision
machine_id,
pc_01,0.1774
pc_02,0.1867
pc_03,0.1869
pc_04,0.1559
pc_05,0.1559
pc_06,0.1862
pc_07,0.1773
pc_08,0.1774
pc_13,0.0960


**5.6.** Объединение и сохранение результатов

In [49]:
results_dir = ROOT / "results" / "model_quality" / COMPONENT
results_dir.mkdir(parents=True, exist_ok=True)

local_quality = local_fitness.join(local_precision, how="left")
global_quality = global_fitness.join(global_precision, how="left")

local_quality.to_csv(results_dir / "local_quality.csv")
local_fitness.to_csv(results_dir / "local_fitness.csv")
local_precision.to_csv(results_dir / "local_precision.csv")
fitness_matrix.to_csv(results_dir / "fitness_matrix.csv")
global_quality.to_csv(results_dir / "global_quality.csv")
global_fitness.to_csv(results_dir / "global_fitness.csv")
global_precision.to_csv(results_dir / "global_precision.csv")

print(f"Таблицы сохранены в: {results_dir}")

Таблицы сохранены в: C:\Users\lena\Desktop\diplom\EquipmentEventLogGeneration\results\model_quality\kernel_power
